## **Cell 1 - Mount Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR  = "/content/drive/MyDrive/MedChatbot_Data_1"
PDF_DIR   = f"{SAVE_DIR}/pdfs"
DATA_DIR  = f"{SAVE_DIR}/table_data"
CLEAN_DIR = f"{SAVE_DIR}/clean_data"
IMG_DIR   = f"{SAVE_DIR}/figures"
LOG_DIR   = f"{SAVE_DIR}/logs"

for folder in [PDF_DIR, DATA_DIR, CLEAN_DIR, IMG_DIR, LOG_DIR]:
    os.makedirs(folder, exist_ok=True)

print("✓ Google Drive mounted")
print(f"✓ All files will be saved to: {SAVE_DIR}")

Mounted at /content/drive
✓ Google Drive mounted
✓ All files will be saved to: /content/drive/MyDrive/MedChatbot_Data_1


## **Cell 2 - Install Dependencies**

In [ ]:
!apt-get update -q
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -q ./google-chrome-stable_current_amd64.deb
!pip install selenium webdriver-manager requests beautifulsoup4 pymupdf pandas Pillow google-generativeai tqdm -q
!pip install pinecone sentence-transformers rank-bm25 -q

print("✓ All dependencies installed")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,385 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,793 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64

## **Cell 3 - Configurations**

In [ ]:
WEBSITES = [
    {
        "name"        : "MOH_Malaysia_CPG",
        "url"         : "https://mymahtas.moh.gov.my/index.php/docman-list/publications/cpg-list",
        "type"        : "javascript",
        "scrape_type" : "pdf",
        "base_url"    : "https://mymahtas.moh.gov.my",
    },
    {
        "name"        : "FUKKM_Drug_Formulary",
        "url"         : "https://pharmacy.moh.gov.my/en/apps/fukkm",
        "type"        : "static",
        "scrape_type" : "table",
        "base_url"    : "https://pharmacy.moh.gov.my",
        "total_pages" : 56,      # pages 0–55 (confirmed from site: last >> ?page=55)
        "page_param"  : "page",
    },
]

# Non-CPG file types to skip during preprocessing
# Uses whole-word matching to avoid false positives (e.g. "INQUIRY" contains "QR")
EXCLUDE_KEYWORDS = ["QR", "PIL", "TM"]

# ── Pinecone config ──────────────────────────────────────────────
PINECONE_API_KEY  = "pcsk_7Nmoc5_AD8WgDfQedS9u8fxdbp9wD5y8Ae9cqVYKLRj2bWy217AYLetj87Vbg8nxm261cR"   # ← paste yours
INDEX_NAME        = "medical-knowledge"

# ── Chunking config ──────────────────────────────────────────────
PARENT_SIZE       = 1500   # words per parent chunk
CHILD_SIZE        = 500    # words per child chunk
OVERLAP           = 100    # overlap between chunks

# ── Embedding config ─────────────────────────────────────────────
EMBEDDING_MODEL   = "pritamdeka/S-PubMedBert-MS-MARCO"
DIMENSION         = 768

# ── Retrieval config ─────────────────────────────────────────────
TOP_K             = 20
BATCH_SIZE        = 32    # embedding batch size

print("✓ Configuration ready")

✓ Configuration ready


## **Cell 4 - Import Libraries**

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from collections import Counter
from pathlib import Path
from tqdm import tqdm
import fitz          # PyMuPDF
import requests
import pandas as pd
import numpy as np
import re, json, time, os, traceback, pickle

print("✓ All libraries imported")

✓ All libraries imported


## **Cell 5 - Helper Functions**

In [ ]:
def create_driver():
    """Create headless Chrome browser for JS-rendered pages."""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.binary_location = "/usr/bin/google-chrome"
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)


def sanitize_filename(name):
    """
    Remove illegal filename characters and normalise whitespace.
    Also strips any existing .pdf extension to prevent double extension.
    """
    name = re.sub(r'\.pdf$', '', name, flags=re.IGNORECASE)  # prevent double .pdf
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = re.sub(r'\s+', '_', name.strip())
    return name[:80]


def resolve_url(href, base_url):
    """
    Convert any href to an absolute URL.
    Handles: absolute, root-relative (/path), and relative (path) links.
    Skips javascript: links.
    """
    if not href:
        return None
    href = href.strip()
    if href.lower().startswith("javascript"):
        return None
    if href.startswith("http"):
        return href
    # urljoin handles both /relative and relative/path correctly
    return urljoin(base_url + "/", href)


def check_robots(base_url):
    """Log robots.txt status. Informational only."""
    try:
        r = requests.get(f"{base_url}/robots.txt", timeout=5)
        if "Disallow: /" in r.text:
            print(f"  ⚠ robots.txt restricts scraping — proceed carefully")
        else:
            print(f"  ✓ robots.txt OK")
    except Exception:
        print(f"  ℹ robots.txt not accessible — proceeding")


def download_with_retry(url, headers, retries=3, timeout=60):
    """
    Download a URL with exponential backoff retry.
    Returns response object or raises on all retries exhausted.
    """
    for attempt in range(retries):
        try:
            resp = requests.get(
                url, headers=headers, timeout=timeout,
                stream=True, allow_redirects=True
            )
            resp.raise_for_status()
            return resp
        except Exception as e:
            if attempt < retries - 1:
                wait = 3 * (attempt + 1)
                print(f"    ⚠ Retry {attempt + 1}/{retries} in {wait}s — {e}")
                time.sleep(wait)
            else:
                raise

print("✓ Helper functions ready")

✓ Helper functions ready


## **Cell 6 - Scraper Functions**

In [ ]:
def scrape_cpg_pdfs(site):
    """
    Scrape CPG PDF links from JS-rendered MOH page using Selenium.
    Only scrapes the CPG column (col index 1) — skips QR/TM/PIL columns.
    """
    driver = None
    try:
        print(f"  [Selenium] Opening browser...")
        driver = create_driver()
        driver.get(site["url"])

        # Wait for actual table data cells to be present (not just the table tag)
        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "table tr td"))
            )
        except Exception:
            print("  ⚠ Table data wait timed out — continuing anyway")

        time.sleep(3)   # extra settle time for JS rendering

        pdf_links = []
        seen      = set()
        tables    = driver.find_elements(By.TAG_NAME, "table")

        for table in tables:
            rows = table.find_elements(By.TAG_NAME, "tr")
            for row in rows:
                cells = row.find_elements(By.TAG_NAME, "td")
                # Table structure: Title(0) | CPG(1) | QR(2) | TM(3) | PIL(4) | Year(5) | Status(6)
                if len(cells) >= 2:
                    title_text = cells[0].text.strip()
                    cpg_cell   = cells[1]   # CPG column only
                    for link in cpg_cell.find_elements(By.TAG_NAME, "a"):
                        href = link.get_attribute("href") or ""
                        href = resolve_url(href, site["base_url"])
                        if href and ".pdf" in href.lower() and href not in seen:
                            name = title_text or href.split("/")[-1]
                            pdf_links.append({
                                "name"  : name,
                                "url"   : href,
                                "source": site["name"],
                            })
                            seen.add(href)
                            print(f"    ✓ {name}")

        return pdf_links

    except Exception as e:
        print(f"  ✗ Selenium error: {e}")
        return []
    finally:
        if driver:
            driver.quit()
            print("  ✓ Browser closed")


def scrape_fukkm_table(site):
    """
    Scrape all 56 pages of FUKKM drug formulary.
    Columns: #(0) | Generic Name(1) | MDC(2) | Category(3) |
             Indications(4) | Pres. Restrictions(5) | Dosage(6)
    Page indexing: 0-based (?page=0 to ?page=55), confirmed from site pagination.
    """
    req_headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120"}
    all_rows    = []
    total       = site["total_pages"]   # 56 → range(56) = 0..55 ✅

    for page_num in range(total):
        url = f"{site['url']}?{site['page_param']}={page_num}"
        print(f"  Scraping page {page_num + 1}/{total}", end="\r")
        try:
            resp = requests.get(url, headers=req_headers, timeout=30)
            resp.raise_for_status()
            soup  = BeautifulSoup(resp.text, "html.parser")
            table = soup.find("table")
            if not table:
                print(f"\n  ⚠ No table found on page {page_num + 1}")
                continue

            for row in table.find_all("tr"):
                cols = row.find_all("td")   # <th> header rows have 0 <td> → skipped ✅
                if len(cols) >= 6:
                    all_rows.append({
                        "no"                     : cols[0].get_text(strip=True),
                        "generic_name"           : cols[1].get_text(strip=True),
                        "mdc_code"               : cols[2].get_text(strip=True),
                        "category"               : cols[3].get_text(strip=True),
                        "indications"            : cols[4].get_text(strip=True),
                        "prescribing_restriction": cols[5].get_text(strip=True),
                        "dosage"                 : cols[6].get_text(strip=True) if len(cols) > 6 else "",
                    })

            time.sleep(1)   # polite delay between pages

        except Exception as e:
            print(f"\n  ✗ Page {page_num + 1} error: {e}")
            continue

    print(f"\n  ✓ Total drug entries scraped: {len(all_rows)}")
    return all_rows

print("✓ Scraper functions ready")

✓ Scraper functions ready


## **Cell 7 - Scrape + Download**

In [ ]:
HTTP_HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120"}

for site in WEBSITES:
    print(f"\n{'='*60}")
    print(f"  {site['name']}")
    print(f"{'='*60}")
    check_robots(site["base_url"])

    # ── CPG PDF site ─────────────────────────────────────────────
    if site["scrape_type"] == "pdf":
        links = scrape_cpg_pdfs(site)
        print(f"\n✓ Found {len(links)} PDF links")

        # Save link manifest for QC
        links_log = f"{LOG_DIR}/{site['name']}_links.json"
        with open(links_log, "w") as f:
            json.dump(links, f, indent=2)
        print(f"✓ Link manifest saved → {links_log}")

        # Download PDFs
        source_folder = f"{PDF_DIR}/{site['name']}"
        os.makedirs(source_folder, exist_ok=True)

        downloaded, failed = [], []

        for i, item in enumerate(links, 1):
            name     = sanitize_filename(item["name"]) or f"doc_{i}"
            filepath = f"{source_folder}/{name}.pdf"

            # Skip if file already exists AND is a valid size (>5 KB)
            # Guards against re-skipping corrupt 0-byte files from a previous crash
            if os.path.exists(filepath) and os.path.getsize(filepath) > 5_000:
                print(f"  [{i:02d}/{len(links)}] ⏭  Already exists: {name}")
                downloaded.append(filepath)
                continue

            print(f"  [{i:02d}/{len(links)}] ⬇  {name}")
            try:
                resp = download_with_retry(item["url"], HTTP_HEADERS)

                # Verify response is actually a PDF (not a redirect HTML page)
                content_type = resp.headers.get("Content-Type", "").lower()
                is_pdf = (
                    "pdf" in content_type
                    or resp.url.lower().endswith(".pdf")
                    or item["url"].lower().endswith(".pdf")
                )
                if not is_pdf:
                    print(f"    ⚠ Unexpected content-type '{content_type}' — skipping")
                    failed.append({**item, "reason": f"bad content-type: {content_type}"})
                    continue

                with open(filepath, "wb") as f:
                    for chunk in resp.iter_content(8192):
                        f.write(chunk)

                size_kb = os.path.getsize(filepath) // 1024
                if size_kb < 5:
                    print(f"    ⚠ File too small ({size_kb} KB) — likely corrupt, removing")
                    os.remove(filepath)
                    failed.append({**item, "reason": "file too small"})
                    continue

                print(f"    ✓ {size_kb} KB saved")
                downloaded.append(filepath)
                time.sleep(1.5)   # polite delay between downloads

            except Exception as e:
                print(f"    ✗ Failed: {e}")
                failed.append({**item, "reason": str(e)})

        print(f"\n  ✓ Downloaded : {len(downloaded)}")
        print(f"  ✗ Failed     : {len(failed)}")

        # Always save failed list — essential for re-running
        if failed:
            failed_log = f"{LOG_DIR}/{site['name']}_failed_downloads.json"
            with open(failed_log, "w") as f:
                json.dump(failed, f, indent=2)
            print(f"  ⚠ Failed log → {failed_log} (re-run these manually)")

    # ── FUKKM table site ─────────────────────────────────────────
    elif site["scrape_type"] == "table":
        rows = scrape_fukkm_table(site)
        if rows:
            raw_path = f"{DATA_DIR}/{site['name']}_raw.json"
            with open(raw_path, "w", encoding="utf-8") as f:
                json.dump(rows, f, indent=2, ensure_ascii=False)
            print(f"  ✓ Raw FUKKM data saved → {raw_path}")
        else:
            print("  ✗ No rows scraped — check connection and try again")

    time.sleep(2)

print("\n✅ All scraping complete")


  MOH_Malaysia_CPG
  ⚠ robots.txt restricts scraping — proceed carefully
  [Selenium] Opening browser...
    ✓ CPG%20Cancer%20pain-19_12_24.pdf
    ✓ CPG_Management_of_Breast_Cancer_(Third_Edition)_.pdf
    ✓ CPG%20Management%20of%20Colorectal%20%20Carcinoma.pdf
    ✓ CPG-Nasopharyngeal%20Carcinoma.pdf
    ✓ CPG%20Management%20of%20Cervical%20Cancer%20(Second%20Edition).pdf
    ✓ CPG%20on%20Dyslipidaemia%202023_6th%20Ed.pdf
    ✓ CPG%20on%20Heart%20Failure%202023_5th%20Ed.pdf
    ✓ CPG_Management_of_NSTE-ACS_3rd_Edition_2021.pdf
    ✓ CPG_Management_of_Ischaemic_Stroke_3rd_Edition_2020_Version_03.04_.2023_(softcopy)_%20(1).pdf
    ✓ MANAGEMENT%20OF%20ACUTE%20ST%20SEGMENT%20ELEVATION%20MYOCARDIAL%20INFARCTION%20(STEM)%202019%20(1).pdf
    ✓ CPG%20Maagement%20of%20Hypertension%20CPG%202018%20V3.8%20FA.pdf
    ✓ CPG%20OF%20Stable%20Coronary%20Artery%20Disease%20(2nd%20Edition).pdf
    ✓ CPG%20on%20Primary%20&%20Secondary%20Prevention%20of%20CVD%202017%20(1).pdf
    ✓ Prevention,%20Diagno

## **Cell 8 - Preprocessing Functions**

In [ ]:
# ── Abbreviation expansion ────────────────────────────────────────
# Only expand abbreviations that are truly unambiguous in ALL contexts.
# Removed: hr (corrupts "1 hr infusion"), bp/bmi/wbc/rbc/hb (no RAG value),
#          rx (not standard Malaysian CPG notation)
MED_ABBREVIATIONS = {
    r'\bpt\b'  : 'patient',
    r'\bpts\b' : 'patients',
    r'\bhx\b'  : 'history',
    r'\bdx\b'  : 'diagnosis',
    r'\btx\b'  : 'treatment',
    r'\bsx\b'  : 'symptoms',
}

# ── Static boilerplate patterns seen across all MOH CPGs ─────────
STATIC_BOILERPLATE = [
    r'All rights reserved[^\n]*\n',
    r'Copyright[^\n]*\n',
    r'Ministry of Health Malaysia[^\n]*\n',
    r'Printed in Malaysia[^\n]*\n',
    r'MaHTAS[^\n]*\n',
    r'Medical Development Division[^\n]*\n',
    r'(?m)^Management of .+\n',    # fallback for repeating CPG title
]

# ── Clinical content — protect from short-line filter ────────────
CLINICAL_KEEP = re.compile(
    r'(\d+\s*mg|\bgrade\s*[a-e]\b|contraindicated|mmhg|'
    r'recommendation\s*\d+|level\s+[ivxIVX]+|\d+\s*mcg|'
    r'\d+\s*ml|clinical\s+question|key\s+message)',
    re.IGNORECASE
)

# ── Word list for garbled page detection ─────────────────────────
# Extended to catch actual reversed words found in MOH CPGs (e.g. page 102
# of Cancer Pain PDF: 'ssertsid'='distress', 'emertxE'='Extreme')
# Also includes Malay words to prevent false positives on bilingual pages
COMMON_WORDS = {
    # English medical
    'pain', 'care', 'dose', 'drug', 'with', 'level', 'score',
    'patient', 'health', 'treatment', 'management', 'distress',
    'extreme', 'none', 'rating', 'form', 'list', 'name', 'date',
    'time', 'note', 'test', 'item', 'scale', 'mild', 'severe',
    'blood', 'pressure', 'clinical', 'disease', 'risk', 'more',
    # Malay medical — prevent false garbled detection
    'pesakit', 'ubat', 'rawatan', 'penyakit', 'dos', 'hospital',
    'kesihatan', 'malaysia', 'kementerian', 'dengan', 'untuk',
}


def is_cpg_file(filename):
    """
    Return True only for main CPG files. Skip QR, PIL, TM variants.
    Uses whole-word matching on filename stem to avoid false positives,
    e.g. "INQUIRY" contains "QR" but should NOT be excluded.
    """
    stem  = Path(filename).stem.upper()
    parts = re.split(r'[\s_\-\.]+', stem)
    return not any(kw.upper() in parts for kw in EXCLUDE_KEYWORDS)


def detect_pdf_type(doc):
    """
    Classify PDF as 'text', 'mixed', or 'scanned'.
    Scanned PDFs need OCR and cannot be preprocessed here.
    Samples first 5 pages to avoid cover-page false positives.
    """
    lengths = [len(page.get_text("text").strip()) for page in list(doc)[:5]]
    avg = sum(lengths) / max(len(lengths), 1)
    if avg < 50:   return "scanned"
    if avg < 200:  return "mixed"
    return "text"


def detect_repeating_headers(doc, sample_pages=10):
    """
    Auto-detect lines that appear on >40% of sampled pages.
    This catches ANY CPG title (e.g. 'Management of Cancer Pain (Second Edition)')
    without hardcoding — works across all 54 different CPG documents.
    """
    line_counts    = Counter()
    pages_to_check = min(sample_pages, len(doc))

    for page in list(doc)[:pages_to_check]:
        for line in page.get_text("text").split('\n'):
            line = line.strip()
            if len(line) > 10:
                line_counts[line] += 1

    threshold = pages_to_check * 0.4
    return {line for line, count in line_counts.items() if count >= threshold}


def is_garbled(text, repeating_headers=None):
    """
    Detect rotated/image pages that extract as reversed text.
    Strips repeating headers first — without this, the normal CPG title
    that appears on every page drowns out the garbled words in the ratio.
    Malay-aware: won't flag valid Malay text as garbled.
    """
    check_text = text
    if repeating_headers:
        for header in repeating_headers:
            check_text = check_text.replace(header, '')

    words = [w for w in check_text.split() if len(w) > 3]
    if not words:
        return False
    hits = sum(1 for w in words if w[::-1].lower() in COMMON_WORDS)
    return (hits / len(words)) > 0.35


def extract_tables_as_text(page):
    """
    Extract tables from a fitz page using Header: Value format.
    This preserves column context for RAG — a cell value like '500mg'
    means nothing without its header 'Morphine dose'.
    Skips ghost tables: image-based tables where cells are all empty.
    """
    tables = []
    try:
        for tbl in page.find_tables():
            extracted = tbl.extract()
            if not extracted or len(extracted) < 2:
                continue

            # Guard: skip image-based tables (all cells empty)
            all_cells   = [c for row in extracted for c in row if c and str(c).strip()]
            total_cells = sum(len(row) for row in extracted)
            if total_cells == 0 or len(all_cells) / total_cells < 0.3:
                continue   # ghost table — skip silently

            headers    = [str(c).strip() if c else "" for c in extracted[0]]
            table_text = "\n[TABLE]\n"

            for row in extracted[1:]:
                pairs = [
                    f"{h}: {str(v).strip()}"
                    for h, v in zip(headers, row)
                    if v and str(v).strip() and h
                ]
                if pairs:
                    table_text += " | ".join(pairs) + "\n"

            table_text += "[END TABLE]\n"
            if table_text.strip() not in ("[TABLE]\n[END TABLE]", ""):
                tables.append(table_text)

    except Exception:
        pass   # table extraction failure is non-fatal
    return tables


def strip_references(text):
    """
    Remove REFERENCES section from the end of a CPG document.
    Only strips if REFERENCES appears in the LAST 20% of the document.
    Original approach (re.DOTALL from start) wiped content after ANY
    mention of 'references' mid-document (e.g. 'refer to references above').
    """
    cutoff = int(len(text) * 0.80)
    match  = re.search(r'\nREFERENCES\b', text[cutoff:], re.IGNORECASE)
    if match:
        return text[:cutoff + match.start()]
    return text


def clean_text(text, repeating_headers=None):
    """
    Clean raw text from a single PDF page.
    Order matters: remove headers first, then fix hyphenation,
    then remove page numbers, then boilerplate, then short lines.
    """
    # 1. Remove auto-detected repeating headers (e.g. the CPG title)
    if repeating_headers:
        for header in repeating_headers:
            text = text.replace(header, '')

    # 2. Fix hyphenated line breaks: "treat-\nment" → "treatment"
    text = re.sub(r'-\n(\w)', r'\1', text)

    # 3. Remove standalone page numbers (1–3 digit lines alone on a line)
    text = re.sub(r'(?m)^\s*\d{1,3}\s*$', '', text)

    # 4. Remove static boilerplate patterns
    for pattern in STATIC_BOILERPLATE:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # 5. Short line filter — but PROTECT clinical content:
    #    "Recommendation 5", "Grade A", "10 mg/kg" are all short but critical
    lines = [
        line for line in text.split('\n')
        if len(line.split()) > 3
        or CLINICAL_KEEP.search(line)
        or line.strip() == ''
    ]
    text = '\n'.join(lines)

    # 6. Normalise whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()


def normalize_medical_text(text):
    """Expand safe medical abbreviations. Only truly unambiguous ones."""
    for pattern, replacement in MED_ABBREVIATIONS.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text


def extract_and_preprocess_pdf(pdf_path):
    """
    Full preprocessing pipeline for a single CPG PDF.
    Steps:
      1. Detect PDF type (bail if scanned)
      2. Auto-detect repeating headers for this specific document
      3. Per-page: skip empty/garbled, clean text, extract tables
      4. Strip references section (last 20% only)
      5. Expand safe medical abbreviations
    Returns: (full_text, log_dict)
    Note: No deduplication — it incorrectly drops valid repeated clinical
          sentences (e.g. multiple recommendations with similar structure)
    """
    filename = os.path.basename(pdf_path)
    log = {
        "file"            : filename,
        "status"          : None,
        "pdf_type"        : None,
        "total_pages"     : 0,
        "processed_pages" : 0,
        "skipped_empty"   : 0,
        "skipped_garbled" : 0,
        "tables_found"    : 0,
        "output_chars"    : 0,
        "warnings"        : [],
        "error"           : None,
    }

    try:
        doc = fitz.open(pdf_path)
        log["total_pages"] = len(doc)

        # Step 1: Detect PDF type
        pdf_type        = detect_pdf_type(doc)
        log["pdf_type"] = pdf_type

        if pdf_type == "scanned":
            log["status"] = "skipped_scanned"
            log["warnings"].append("Fully scanned PDF — needs OCR, skipped")
            doc.close()
            return None, log

        if pdf_type == "mixed":
            log["warnings"].append("Mixed PDF — some pages may be image-based")

        # Step 2: Auto-detect this document's specific repeating header
        repeating_headers = detect_repeating_headers(doc)
        if repeating_headers:
            log["warnings"].append(f"Auto-removed headers: {repeating_headers}")

        # Step 3: Process each page
        parts = []
        for page_num, page in enumerate(doc):
            raw_text = page.get_text("text")

            # Skip: empty pages (covers, back pages, section dividers)
            if len(raw_text.strip()) < 30:
                log["skipped_empty"] += 1
                continue

            # Skip: rotated/image pages that extract as garbled text
            if is_garbled(raw_text, repeating_headers):
                log["skipped_garbled"] += 1
                log["warnings"].append(f"Garbled page skipped: pg {page_num + 1}")
                continue

            cleaned    = clean_text(raw_text, repeating_headers)
            table_txts = extract_tables_as_text(page)
            log["tables_found"] += len(table_txts)

            page_content = cleaned
            if table_txts:
                page_content += "\n" + "\n".join(table_txts)

            if page_content.strip():
                parts.append(page_content)
                log["processed_pages"] += 1

        doc.close()

        # Step 4–5: Assemble, strip references, expand abbreviations
        full_text = "\n\n".join(parts)
        full_text = strip_references(full_text)
        full_text = normalize_medical_text(full_text)

        log["output_chars"] = len(full_text)
        log["status"]       = "success"

        if log["output_chars"] < 1000:
            log["warnings"].append("⚠ Very little text extracted — verify manually")

        return full_text, log

    except Exception as e:
        log["status"] = "error"
        log["error"]  = traceback.format_exc()
        return None, log

print("✓ PDF preprocessing functions ready")

✓ PDF preprocessing functions ready


## **Cell 9 - Preprocessing across all CPG PDFs**

In [ ]:
cpg_folder = f"{PDF_DIR}/MOH_Malaysia_CPG"
all_pdfs   = list(Path(cpg_folder).glob("*.pdf"))

# Separate CPG files from QR/PIL/TM variants using whole-word matching
cpg_files = [f for f in all_pdfs if is_cpg_file(f.name)]
skipped   = [f for f in all_pdfs if not is_cpg_file(f.name)]

print(f"Total PDFs found : {len(all_pdfs)}")
print(f"CPG files to use : {len(cpg_files)}")
print(f"Non-CPG skipped  : {len(skipped)}")
for f in skipped:
    print(f"  ⏭ {f.name}")

cpg_records = []
all_logs    = []

for i, pdf_path in enumerate(tqdm(sorted(cpg_files)), 1):
    print(f"\n[{i:02d}/{len(cpg_files)}] {pdf_path.name}")

    text, log = extract_and_preprocess_pdf(str(pdf_path))
    all_logs.append(log)

    icon = {"success": "✅", "skipped_scanned": "🖼", "error": "❌"}.get(log["status"], "⚠")
    print(f"  {icon} Status: {log['status']}")

    if log["status"] == "success":
        print(
            f"  Pages: {log['processed_pages']}/{log['total_pages']} | "
            f"Garbled skipped: {log['skipped_garbled']} | "
            f"Tables: {log['tables_found']} | "
            f"Chars: {log['output_chars']:,}"
        )
        cpg_records.append({
            "title"     : pdf_path.stem,
            "filename"  : pdf_path.name,
            "source"    : "MOH_Malaysia_CPG",
            "text"      : text,              # full text — no truncation
            "word_count": len(text.split()),
            "char_count": len(text),
            "type"      : "clinical_guideline",
        })
    elif log["status"] == "skipped_scanned":
        print(f"  🖼 Scanned PDF — needs OCR before it can join the RAG database")
    else:
        print(f"  Error: {str(log.get('error', ''))[:120]}")

    for w in log["warnings"]:
        print(f"  ⚠ {w}")

# Save preprocessing log for QC review
log_path = f"{LOG_DIR}/cpg_preprocessing_log.json"
with open(log_path, "w") as f:
    json.dump(all_logs, f, indent=2)

# Print batch summary
success = sum(1 for l in all_logs if l["status"] == "success")
scanned = sum(1 for l in all_logs if l["status"] == "skipped_scanned")
errors  = sum(1 for l in all_logs if l["status"] == "error")

print(f"\n{'='*50}")
print(f"✅ Success : {success}")
print(f"🖼 Scanned : {scanned}  ← these need OCR to be usable")
print(f"❌ Errors  : {errors}")
print(f"📋 Full log → {log_path}")
print(f"📄 Total words preprocessed: {sum(r['word_count'] for r in cpg_records):,}")

Total PDFs found : 54
CPG files to use : 53
Non-CPG skipped  : 1
  ⏭ QR-Schizophrenia_11012023%20(1).pdf


  0%|          | 0/53 [00:00<?, ?it/s]


[01/53] CPG%20ABX.pdf
Consider using the pymupdf_layout package for a greatly improved page layout analysis.


  2%|▏         | 1/53 [00:06<05:32,  6.39s/it]

  ✅ Status: success
  Pages: 79/81 | Garbled skipped: 0 | Tables: 21 | Chars: 133,751
  ⚠ Auto-removed headers: {'Antibiotic Prophylaxis in Oral and Maxillofacial Surgery for', 'Prevention of Surgical Site Infection (3rd Edition)'}

[02/53] CPG%20Cancer%20pain-19_12_24.pdf


...

  ✅ Status: success
  Pages: 94/96 | Garbled skipped: 0 | Tables: 31 | Chars: 171,046
  ⚠ Auto-removed headers: {'Management of Gout (Second Edition)'}

[52/53] e-CPG_Management_of_Retinopathy_of_Prematurity_-4_10_23.pdf


 98%|█████████▊| 52/53 [10:18<00:07,  7.89s/it]

  ✅ Status: success
  Pages: 70/74 | Garbled skipped: 0 | Tables: 17 | Chars: 118,711
  ⚠ Auto-removed headers: {'Management of Retinopathy of Prematurity (Second Edition)'}

[53/53] eCPG%20BD2_24102025.pdf


100%|██████████| 53/53 [10:47<00:00, 12.22s/it]

  ✅ Status: success
  Pages: 92/92 | Garbled skipped: 0 | Tables: 27 | Chars: 212,492
  ⚠ Auto-removed headers: {'CLINICAL PRACTICE GUIDELINES', 'Psychiatrist', 'MANAGEMENT OF ERECTILE DYSFUNCTION'}

✅ Success : 52
🖼 Scanned : 1  ← these need OCR to be usable
❌ Errors  : 0
📋 Full log → /content/drive/MyDrive/MedChatbot_Data_1/logs/cpg_preprocessing_log.json
📄 Total words preprocessed: 1,712,054


## **Cell 10 - Process FUKKM**

In [ ]:
def preprocess_fukkm(raw_json_path):
    """
    Clean and normalise FUKKM drug formulary rows.
    Each row becomes one record ready for RAG embedding.
    combined_text includes mdc_code — needed for drug lookup by code.
    No chunking needed: each drug row is already a self-contained unit.
    """
    with open(raw_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    cleaned = []
    for row in data:
        if not row.get("generic_name", "").strip():
            continue   # skip empty rows

        clean_row = {}
        for key, val in row.items():
            # Normalise internal whitespace and newlines; str() for safety
            val = re.sub(r'\s+', ' ', str(val)).strip()
            clean_row[key] = val

        # combined_text — the field that will be embedded for RAG retrieval
        # mdc_code included so chatbot can answer "what is the MDC code for X"
        clean_row["combined_text"] = (
            f"Drug: {clean_row.get('generic_name', '')}. "
            f"MDC Code: {clean_row.get('mdc_code', '')}. "
            f"Category: {clean_row.get('category', '')}. "
            f"Indications: {clean_row.get('indications', '')}. "
            f"Prescribing Restrictions: {clean_row.get('prescribing_restriction', '')}. "
            f"Dosage: {clean_row.get('dosage', '')}."
        )
        clean_row["source"] = "FUKKM_Drug_Formulary"
        clean_row["type"]   = "drug_formulary"
        cleaned.append(clean_row)

    return cleaned


raw_fukkm_path = f"{DATA_DIR}/FUKKM_Drug_Formulary_raw.json"

if os.path.exists(raw_fukkm_path):
    fukkm_records = preprocess_fukkm(raw_fukkm_path)
    print(f"✅ Preprocessed {len(fukkm_records)} drug entries")
    if fukkm_records:
        print(f"\nSample record:")
        print(json.dumps(fukkm_records[0], indent=2, ensure_ascii=False))
else:
    print("⚠ FUKKM raw data not found — run Cell 7 first")
    fukkm_records = []

✅ Preprocessed 1675 drug entries

Sample record:
{
  "no": "1",
  "generic_name": "2-deoxy-2-[18F] fluoro-D-glucose [18F] FDG Injection",
  "mdc_code": "V09IX04-000-P30-02-XXX",
  "category": "A*",
  "indications": "Indicated for positron emission tomography (PET) imaging in the following setting:i. Oncology: For assessment of abnormal glucose metabolism to assist in the evaluation of malignancy in patients with known or suspected abnormalities found by other testing modalities, or in patients with an existing diagnosis of cancerii. Cardiology: For the identification of left ventricular myocardium with residual glucose metabolism and reversible loss of systolic function in patients with coronary artery disease and left ventricular dysfunction, when used together with myocardial perfusion imaging.iii. Neurology: For the identification of regions of abnormal glucose metabolism associated with foci or epileptic seizures",
  "prescribing_restriction": "To be prescribed by Nuclear Medicine 

## **Cell 11 - Save All Clean dataset**

In [ ]:
# ── Save CPG clean data ──────────────────────────────────────────
if cpg_records:
    cpg_path = f"{CLEAN_DIR}/cpg_clean.json"
    with open(cpg_path, "w", encoding="utf-8") as f:
        json.dump(cpg_records, f, indent=2, ensure_ascii=False)
    print(f"✓ CPG data saved → {cpg_path}")
    print(f"  Documents : {len(cpg_records)}")
    print(f"  Total words: {sum(r['word_count'] for r in cpg_records):,}")

# ── Save FUKKM clean data ─────────────────────────────────────────
if fukkm_records:
    fukkm_path = f"{CLEAN_DIR}/fukkm_clean.json"
    with open(fukkm_path, "w", encoding="utf-8") as f:
        json.dump(fukkm_records, f, indent=2, ensure_ascii=False)
    print(f"\n✓ FUKKM data saved → {fukkm_path}")
    print(f"  Drug entries: {len(fukkm_records)}")

# ── Build & save combined dataset ────────────────────────────────
# Consistent schema across both sources:
# id | source | title | text | type
# 'text' is the field your RAG chunker will read — same name for both sources
cpg_combined = [
    {
        "id"    : f"cpg_{i}",
        "source": r["source"],
        "title" : r["title"],
        "text"  : r["text"],       # full text — no truncation
        "type"  : r["type"],
    }
    for i, r in enumerate(cpg_records)
]

fukkm_combined = [
    {
        "id"    : f"fukkm_{i}",
        "source": r["source"],
        "title" : r.get("generic_name", ""),
        "text"  : r["combined_text"],   # same 'text' key as CPG ✅
        "type"  : r["type"],
    }
    for i, r in enumerate(fukkm_records)
]

all_data      = cpg_combined + fukkm_combined
combined_path = f"{CLEAN_DIR}/medchatbot_dataset.json"

with open(combined_path, "w", encoding="utf-8") as f:
    json.dump(all_data, f, indent=2, ensure_ascii=False)

print(f"\n{'='*50}")
print(f"✅ Combined dataset saved → {combined_path}")
print(f"   CPG documents : {len(cpg_combined)}")
print(f"   Drug entries  : {len(fukkm_combined)}")
print(f"   Total records : {len(all_data)}")
print(f"\nSchema of each record:")
print(f"  id     — unique identifier (cpg_0, fukkm_0, ...)")
print(f"  source — MOH_Malaysia_CPG | FUKKM_Drug_Formulary")
print(f"  title  — document/drug name")
print(f"  text   — full clean text ready for RAG chunking")
print(f"  type   — clinical_guideline | drug_formulary")

✓ CPG data saved → /content/drive/MyDrive/MedChatbot_Data_1/clean_data/cpg_clean.json
  Documents : 52
  Total words: 1,712,054

✓ FUKKM data saved → /content/drive/MyDrive/MedChatbot_Data_1/clean_data/fukkm_clean.json
  Drug entries: 1675

✅ Combined dataset saved → /content/drive/MyDrive/MedChatbot_Data_1/clean_data/medchatbot_dataset.json
   CPG documents : 52
   Drug entries  : 1675
   Total records : 1727

Schema of each record:
  id     — unique identifier (cpg_0, fukkm_0, ...)
  source — MOH_Malaysia_CPG | FUKKM_Drug_Formulary
  title  — document/drug name
  text   — full clean text ready for RAG chunking
  type   — clinical_guideline | drug_formulary


## **Cell 12 - Quality Check**

In [ ]:
print("=" * 55)
print("QUALITY CHECK")
print("=" * 55)

# CPG check
print(f"\n📋 CPG Documents ({len(cpg_records)}):")
for r in cpg_records:
    est_tokens = int(r["word_count"] * 1.3)
    status     = "✅" if r["word_count"] > 500 else "⚠ Too short"
    print(f"  {status} {r['title'][:50]}")
    print(f"       Words: {r['word_count']:,} | Est. tokens: {est_tokens:,}")

# FUKKM check
print(f"\n💊 FUKKM Drug Entries : {len(fukkm_records)}")
empty = [r for r in fukkm_records if not r.get("indications")]
print(f"  ⚠ Entries with no indications: {len(empty)}")

# File summary
print(f"\n📁 Files saved in Drive:")
print(f"  {CLEAN_DIR}/cpg_clean.json")
print(f"  {CLEAN_DIR}/fukkm_clean.json")
print(f"  {CLEAN_DIR}/medchatbot_dataset.json  ← use this for Pinecone")
print(f"  {LOG_DIR}/                           ← scrape logs")

print(f"\n✅ Data is ready for chunking & embedding!")

QUALITY CHECK

📋 CPG Documents (52):
  ✅ CPG%20ABX
       Words: 19,073 | Est. tokens: 24,794
  ✅ CPG%20Cancer%20pain-19_12_24
       Words: 29,918 | Est. tokens: 38,893
  ✅ CPG%20Dengue%20Infection%20PDF%20Final%20(1)
       Words: 23,428 | Est. tokens: 30,456
  ✅ CPG%20ECC
       Words: 29,881 | Est. tokens: 38,845
  ✅ CPG%20Early%20Management%20of%20Head%20Injury%20in
       Words: 22,743 | Est. tokens: 29,565
  ✅ CPG%20Maagement%20of%20Hypertension%20CPG%202018%2
       Words: 51,549 | Est. tokens: 67,013
  ✅ CPG%20Management%20of%20Cervical%20Cancer%20(Secon
       Words: 21,932 | Est. tokens: 28,511
  ✅ CPG%20Management%20of%20Chronic%20Kidney%20%20Dise
       Words: 16,661 | Est. tokens: 21,659
  ✅ CPG%20Management%20of%20Colorectal%20%20Carcinoma
       Words: 21,313 | Est. tokens: 27,706
  ✅ CPG%20Management%20of%20Diabetes%20in%20Pregnancy%
       Words: 17,296 | Est. tokens: 22,484
  ✅ CPG%20Management%20of%20Diabetic%20Foot%20%20(Seco
       Words: 17,124 | Est. tokens: 22,

## **Cell 13 - Load Dataset for Chuncking**

In [ ]:
DATASET_PATH = f"{CLEAN_DIR}/medchatbot_dataset.json"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

cpg_docs  = [d for d in data if d["type"] == "clinical_guideline"]
drug_docs = [d for d in data if d["type"] == "drug_formulary"]

print(f"✓ Dataset loaded from: {DATASET_PATH}")
print(f"  CPG documents : {len(cpg_docs)}")
print(f"  Drug entries  : {len(drug_docs)}")
print(f"  Total         : {len(data)}")

# Quick sanity check
if cpg_docs:
    print(f"\nSample CPG record keys  : {list(cpg_docs[0].keys())}")
if drug_docs:
    print(f"Sample Drug record keys : {list(drug_docs[0].keys())}")

✓ Dataset loaded from: /content/drive/MyDrive/MedChatbot_Data_1/clean_data/medchatbot_dataset.json
  CPG documents : 52
  Drug entries  : 1675
  Total         : 1727

Sample CPG record keys  : ['id', 'source', 'title', 'text', 'type']
Sample Drug record keys : ['id', 'source', 'title', 'text', 'type']


## **Cell 14 - Parent-Child Chunking**

In [ ]:
def create_parent_chunks(text, title, source, doc_type):
    """Split document into large parent chunks (~1500 words)."""
    words   = text.split()
    parents = []
    step    = PARENT_SIZE - OVERLAP
    i       = 0
    p_num   = 0

    while i < len(words):
        chunk_words = words[i:i + PARENT_SIZE]
        chunk_text  = " ".join(chunk_words)

        if len(chunk_words) > 50:
            # Clean title for ID
            clean_title = re.sub(r'[^a-zA-Z0-9]', '_', title[:20])
            parent_id   = f"parent_{doc_type[:3]}_{clean_title}_{p_num}"

            parents.append({
                "parent_id"  : parent_id,
                "title"      : title,
                "source"     : source,
                "type"       : doc_type,
                "text"       : chunk_text,
                "word_count" : len(chunk_words),
                "chunk_num"  : p_num
            })
            p_num += 1

        i += step

    return parents


def create_child_chunks(parent):
    """Split parent into smaller child chunks (~500 words)."""
    words    = parent["text"].split()
    children = []
    step     = CHILD_SIZE - OVERLAP
    i        = 0
    c_num    = 0

    while i < len(words):
        chunk_words = words[i:i + CHILD_SIZE]
        chunk_text  = " ".join(chunk_words)

        if len(chunk_words) > 30:
            child_id = f"child_{parent['parent_id']}_{c_num}"
            children.append({
                "child_id"   : child_id,
                "parent_id"  : parent["parent_id"],  # ← link to parent
                "title"      : parent["title"],
                "source"     : parent["source"],
                "type"       : parent["type"],
                "text"       : chunk_text,
                "word_count" : len(chunk_words),
                "chunk_num"  : c_num
            })
            c_num += 1

        i += step

    return children


def build_parent_child_chunks(cpg_docs, drug_docs):
    """Build all parent + child chunks for entire dataset."""
    all_parents  = {}  # parent_id → parent data
    all_children = []  # flat list of all children

    # ── CPG documents — full parent-child chunking ────────────
    print("Building chunks for CPG documents...")
    for doc in tqdm(cpg_docs):
        parents = create_parent_chunks(
            text     = doc["text"],       # consistent "text" key ✅
            title    = doc["title"],
            source   = doc["source"],
            doc_type = "clinical_guideline"
        )
        for parent in parents:
            all_parents[parent["parent_id"]] = parent
            children = create_child_chunks(parent)
            all_children.extend(children)

        doc_children = [c for c in all_children
                        if doc["title"] in c["parent_id"]]
        print(f"  ✓ {doc['title'][:45]}")
        print(f"     Parents: {len(parents)} | "
              f"Children: {len(doc_children)}")

    # ── Drug entries — already short, same text for both ──────
    print(f"\nProcessing drug entries...")
    for i, doc in enumerate(tqdm(drug_docs)):
        clean_title = re.sub(r'[^a-zA-Z0-9]', '_', doc["title"][:20])
        parent_id   = f"parent_drug_{clean_title}_{i}"

        # Parent
        all_parents[parent_id] = {
            "parent_id"  : parent_id,
            "title"      : doc["title"],
            "source"     : doc["source"],
            "type"       : "drug_formulary",
            "text"       : doc["text"],          # consistent "text" key ✅
            "word_count" : len(doc["text"].split()),
            "chunk_num"  : 0
        }

        # Child = same text (drug entries already short)
        child_id = f"child_{parent_id}_0"
        all_children.append({
            "child_id"   : child_id,
            "parent_id"  : parent_id,
            "title"      : doc["title"],
            "source"     : doc["source"],
            "type"       : "drug_formulary",
            "text"       : doc["text"],
            "word_count" : len(doc["text"].split()),
            "chunk_num"  : 0
        })

    return all_parents, all_children


# ── Run chunking ──────────────────────────────────────────────
all_parents, all_children = build_parent_child_chunks(
    cpg_docs, drug_docs
)

# ── Stats ─────────────────────────────────────────────────────
cpg_children  = [c for c in all_children
                 if c["type"] == "clinical_guideline"]
drug_children = [c for c in all_children
                 if c["type"] == "drug_formulary"]

print(f"\n{'='*55}")
print(f"✅ Parent-Child chunking complete!")
print(f"   Total parents       : {len(all_parents)}")
print(f"   Total children      : {len(all_children)}")
print(f"   CPG children        : {len(cpg_children)}")
print(f"   Drug children       : {len(drug_children)}")
print(f"   Avg children/parent : "
      f"{len(all_children)/max(len(all_parents),1):.1f}")

# ── Save to Drive ─────────────────────────────────────────────
parents_path  = f"{CLEAN_DIR}/parents.json"
children_path = f"{CLEAN_DIR}/children.json"

with open(parents_path, "w", encoding="utf-8") as f:
    json.dump(all_parents, f, indent=2, ensure_ascii=False)

with open(children_path, "w", encoding="utf-8") as f:
    json.dump(all_children, f, indent=2, ensure_ascii=False)

print(f"\n✓ Saved to Drive:")
print(f"  {parents_path}")
print(f"  {children_path}")


Building chunks for CPG documents...


 19%|█▉        | 10/52 [00:00<00:00, 97.46it/s]

  ✓ CPG%20ABX
     Parents: 14 | Children: 0
  ✓ CPG%20Cancer%20pain-19_12_24
     Parents: 22 | Children: 0
  ✓ CPG%20Dengue%20Infection%20PDF%20Final%20(1)
     Parents: 17 | Children: 0
  ✓ CPG%20ECC
     Parents: 22 | Children: 0
  ✓ CPG%20Early%20Management%20of%20Head%20Injury
     Parents: 17 | Children: 0
  ✓ CPG%20Maagement%20of%20Hypertension%20CPG%202
     Parents: 37 | Children: 0
  ✓ CPG%20Management%20of%20Cervical%20Cancer%20(
     Parents: 16 | Children: 0
  ✓ CPG%20Management%20of%20Chronic%20Kidney%20%2
     Parents: 12 | Children: 0
  ✓ CPG%20Management%20of%20Colorectal%20%20Carci
     Parents: 16 | Children: 0
  ✓ CPG%20Management%20of%20Diabetes%20in%20Pregn
     Parents: 13 | Children: 0
  ✓ CPG%20Management%20of%20Diabetic%20Foot%20%20
     Parents: 13 | Children: 0
  ✓ CPG%20Management%20of%20Drug%20Resistant%20TB
     Parents: 30 | Children: 0
  ✓ CPG%20Management%20of%20Glaucoma%20(Second%20
     Parents: 15 | Children: 0
  ✓ CPG%20Management%20of%20Multiple%

 58%|█████▊    | 30/52 [00:00<00:00, 82.28it/s]

  ✓ CPG%20on%20Heart%20Failure%202023_5th%20Ed
     Parents: 70 | Children: 0
  ✓ CPG%20on%20Primary%20&%20Secondary%20Preventi
     Parents: 40 | Children: 0
  ✓ CPG-Nasopharyngeal%20Carcinoma
     Parents: 8 | Children: 0
  ✓ CPG-_Management_of_Tuberculosis_(4th_Edition)
     Parents: 25 | Children: 0
  ✓ CPG_Anterior_Crossbite
     Parents: 9 | Children: 0
  ✓ CPG_Management_of_Attention-Deficit_Hyperacti
     Parents: 11 | Children: 0
  ✓ CPG_Management_of_Autism_Spectrum_Disorder_in
     Parents: 15 | Children: 0
  ✓ CPG_Management_of_Breast_Cancer_(Third_Editio
     Parents: 25 | Children: 0
  ✓ CPG_Management_of_Chronic_Hepatitis_C_in_Adul
     Parents: 10 | Children: 0
  ✓ CPG_Management_of_Dengue_in_Children_06072021
     Parents: 13 | Children: 0
  ✓ CPG_Management_of_Ischaemic_Stroke_3rd_Editio
     Parents: 43 | Children: 0
  ✓ CPG_Management_of_MDD_(Second_Edition)_040920
     Parents: 22 | Children: 0
  ✓ CPG_Management_of_Menopause_2022_e-version-1
     Parents: 33 | Chi

...

100%|██████████| 52/52 [00:00<00:00, 71.88it/s]


  ✓ e-CPG_Management_of_Retinopathy_of_Prematurit
     Parents: 13 | Children: 0
  ✓ eCPG%20BD2_24102025
     Parents: 23 | Children: 0

Processing drug entries...


100%|██████████| 1675/1675 [00:00<00:00, 48484.55it/s]


✅ Parent-Child chunking complete!
   Total parents       : 2698
   Total children      : 6579
   CPG children        : 4904
   Drug children       : 1675
   Avg children/parent : 2.4



✓ Saved to Drive:
  /content/drive/MyDrive/MedChatbot_Data_1/clean_data/parents.json
  /content/drive/MyDrive/MedChatbot_Data_1/clean_data/children.json


## **Cell 15 - Build BM25 Index**

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize_for_bm25(text: str) -> list:
    """Tokenize text for BM25 index."""
    text   = text.lower()
    tokens = re.findall(r'\b[a-zA-Z][a-zA-Z0-9]*\b', text)
    stops  = {
        "the", "a", "an", "is", "in", "on", "at", "to",
        "for", "of", "and", "or", "with", "that", "this",
        "are", "was", "were", "be", "been", "has", "have",
        "it", "as", "by", "from", "not", "but", "also"
    }
    return [t for t in tokens if t not in stops and len(t) > 1]

print("Building BM25 index over child chunks...")
print(f"Total children to index: {len(all_children)}\n")

# Tokenize all child chunks
child_texts     = [c["text"]     for c in all_children]
child_ids       = [c["child_id"] for c in all_children]
tokenized_texts = [tokenize_for_bm25(text)
                   for text in tqdm(child_texts,
                   desc="Tokenizing")]

# Build BM25
print("\nBuilding BM25Okapi index...")
bm25 = BM25Okapi(tokenized_texts)

# Save as pickle
bm25_data = {
    "bm25"            : bm25,
    "child_ids"       : child_ids,
    "child_texts"     : child_texts,
    "tokenized_texts" : tokenized_texts
}

bm25_path = f"{CLEAN_DIR}/bm25_index.pkl"
with open(bm25_path, "wb") as f:
    pickle.dump(bm25_data, f)

print(f"\n✅ BM25 index complete!")
print(f"   Indexed documents : {len(child_texts)}")
print(f"   Saved to          : {bm25_path}")

# Quick test
test_query  = "metformin diabetes treatment"
test_tokens = tokenize_for_bm25(test_query)
scores      = bm25.get_scores(test_tokens)
top_idx     = np.argsort(scores)[::-1][:3]

print(f"\nBM25 test — query: '{test_query}'")
for i, idx in enumerate(top_idx):
    print(f"  [{i+1}] score={scores[idx]:.3f} | "
          f"{child_texts[idx][:80]}...")


Building BM25 index over child chunks...
Total children to index: 6579



Tokenizing: 100%|██████████| 6579/6579 [00:01<00:00, 6266.62it/s]



Building BM25Okapi index...

✅ BM25 index complete!
   Indexed documents : 6579
   Saved to          : /content/drive/MyDrive/MedChatbot_Data_1/clean_data/bm25_index.pkl

BM25 test — query: 'metformin diabetes treatment'
  [1] score=14.164 | Drug: Vildagliptin 50 mg and Metformin HCl 500 mg Tablet. MDC Code: A10BD08926T1...
  [2] score=13.991 | Drug: Vildagliptin 50 mg and Metformin HCl 850 mg Tablet. MDC Code: A10BD08926T1...
  [3] score=13.991 | Drug: Vildagliptin 50 mg and Metformin HCl 1000 mg Tablet. MDC Code: A10BD08926T...


## **Cell 16 - Load S-PubMedBert Embedding Model**

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"  GPU : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory
    print(f"  VRAM: {vram/1e9:.1f} GB")

# Load model
print(f"\nLoading {EMBEDDING_MODEL}...")
embedder = SentenceTransformer(EMBEDDING_MODEL, device=device)

# Verify dimension
test_vec  = embedder.encode(["test medical query"])
DIMENSION = test_vec.shape[1]

print(f"\n✅ Embedding model loaded!")
print(f"   Model     : {EMBEDDING_MODEL}")
print(f"   Dimension : {DIMENSION}")
print(f"   Device    : {device}")

# Estimate time
est_sec = (len(all_children) / BATCH_SIZE) * (
    0.5 if device == "cuda" else 2.0
)
print(f"\n   Children to embed  : {len(all_children)}")
print(f"   Estimated time     : "
      f"~{int(est_sec//60)}min {int(est_sec%60)}s on {device}")

Device: cuda
  GPU : Tesla T4
  VRAM: 15.6 GB

Loading pritamdeka/S-PubMedBert-MS-MARCO...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


✅ Embedding model loaded!
   Model     : pritamdeka/S-PubMedBert-MS-MARCO
   Dimension : 768
   Device    : cuda

   Children to embed  : 6579
   Estimated time     : ~1min 42s on cuda


## **Cell 17 - Setup Pinecone Index**

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

# Check existing indexes
existing = [idx.name for idx in pc.list_indexes()]
print(f"Existing indexes: {existing}")

if INDEX_NAME not in existing:
    print(f"\nCreating index '{INDEX_NAME}'...")
    pc.create_index(
        name      = INDEX_NAME,
        dimension = DIMENSION,
        metric    = "dotproduct",   # required for hybrid search
        spec      = ServerlessSpec(
            cloud  = "aws",
            region = "us-east-1"    # free tier region
        )
    )
    print(f"✓ Index created — waiting 30s to initialize...")
    time.sleep(30)
else:
    print(f"✓ Index '{INDEX_NAME}' already exists")

# Connect
index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()

print(f"\n✅ Pinecone ready!")
print(f"   Index name     : {INDEX_NAME}")
print(f"   Dimension      : {DIMENSION}")
print(f"   Metric         : dotproduct")
print(f"   Current vectors: {stats['total_vector_count']}")

Existing indexes: ['medical-knowledge']
✓ Index 'medical-knowledge' already exists

✅ Pinecone ready!
   Index name     : medical-knowledge
   Dimension      : 768
   Metric         : dotproduct
   Current vectors: 0


## **Cell 17 - Load S-PubMedBert Embedding Model**

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"  GPU : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory
    print(f"  VRAM: {vram/1e9:.1f} GB")

# Load model
print(f"\nLoading {EMBEDDING_MODEL}...")
embedder = SentenceTransformer(EMBEDDING_MODEL, device=device)

# Verify dimension
test_vec  = embedder.encode(["test medical query"])
DIMENSION = test_vec.shape[1]

print(f"\n✅ Embedding model loaded!")
print(f"   Model     : {EMBEDDING_MODEL}")
print(f"   Dimension : {DIMENSION}")
print(f"   Device    : {device}")
print(f"   No API key needed ✅")
print(f"   No rate limits   ✅")

# Estimate time
est_sec = (len(all_children) / BATCH_SIZE) * (
    0.5 if device == "cuda" else 2.0
)
print(f"\n   Children to embed  : {len(all_children)}")
print(f"   Estimated time     : "
      f"~{int(est_sec//60)}min {int(est_sec%60)}s on {device}")

Device: cuda
  GPU : Tesla T4
  VRAM: 15.6 GB

Loading pritamdeka/S-PubMedBert-MS-MARCO...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/S-PubMedBert-MS-MARCO
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ Embedding model loaded!
   Model     : pritamdeka/S-PubMedBert-MS-MARCO
   Dimension : 768
   Device    : cuda
   No API key needed ✅
   No rate limits   ✅

   Children to embed  : 6579
   Estimated time     : ~1min 42s on cuda


## **Cell 18 - Embed Children + Upsert to Pinecone**

In [ ]:
# ── Resume tracker ────────────────────────────────────────────
PROGRESS_PATH = f"{LOG_DIR}/embed_progress.json"

def load_progress():
    if os.path.exists(PROGRESS_PATH):
        with open(PROGRESS_PATH, "r") as f:
            return set(json.load(f).get("completed_ids", []))
    return set()

def save_progress(completed_ids: set):
    with open(PROGRESS_PATH, "w") as f:
        json.dump({"completed_ids": list(completed_ids)}, f)

# ── Filter remaining ──────────────────────────────────────────
completed_ids = load_progress()
remaining     = [c for c in all_children
                 if c["child_id"] not in completed_ids]

print(f"Total children     : {len(all_children)}")
print(f"Already uploaded   : {len(completed_ids)}")
print(f"Remaining          : {len(remaining)}")
print(f"\nEmbedding + uploading to Pinecone...\n")

# ── Embed + upsert in batches ─────────────────────────────────
PINECONE_BATCH = 100   # vectors per upsert call
failed_batches = []

for i in tqdm(range(0, len(remaining), BATCH_SIZE),
              desc="Embedding"):
    batch = remaining[i:i + BATCH_SIZE]
    texts = [c["text"] for c in batch]

    # ── Embed locally ─────────────────────────────────────────
    embeddings = embedder.encode(
        texts,
        show_progress_bar    = False,
        convert_to_numpy     = True,
        normalize_embeddings = True,
        batch_size           = BATCH_SIZE
    ).tolist()

    # ── Format for Pinecone ───────────────────────────────────
    vectors = []
    for chunk, embedding in zip(batch, embeddings):
        vectors.append({
            "id"    : chunk["child_id"],
            "values": embedding,
            "metadata": {
                "parent_id" : chunk["parent_id"],
                "title"     : chunk["title"],
                "source"    : chunk["source"],
                "type"      : chunk["type"],
                "text"      : chunk["text"][:800],
                "word_count": chunk["word_count"],
                "chunk_num" : chunk["chunk_num"],
                "chunk_type": "child"
            }
        })

    # ── Upsert to Pinecone ────────────────────────────────────
    try:
        for j in range(0, len(vectors), PINECONE_BATCH):
            sub_batch = vectors[j:j + PINECONE_BATCH]
            index.upsert(vectors=sub_batch, namespace="")

        for chunk in batch:
            completed_ids.add(chunk["child_id"])
        save_progress(completed_ids)

    except Exception as e:
        print(f"\n  ✗ Batch {i} failed: {e}")
        failed_batches.append(i)

    time.sleep(0.2)

# ── Final stats ───────────────────────────────────────────────
stats = index.describe_index_stats()
print(f"\n{'='*55}")
print(f"✅ Children uploaded!")
print(f"   Uploaded      : {len(completed_ids)}")
print(f"   Failed batches: {len(failed_batches)}")
print(f"   Pinecone count: {stats['total_vector_count']}")

if failed_batches:
    print(f"\n⚠ Re-run this cell to retry failed batches")

Total children     : 6579
Already uploaded   : 0
Remaining          : 6579

Embedding + uploading to Pinecone...



Embedding: 100%|██████████| 206/206 [04:45<00:00,  1.39s/it]


✅ Children uploaded!
   Uploaded      : 5708
   Failed batches: 0
   Pinecone count: 5708


## **Cell 19 - Upsert Parents Chunks to Pinecone**

In [ ]:
print(f"Upserting {len(all_parents)} parent chunks...")
print(f"Namespace: 'parents'\n")

parent_list    = list(all_parents.values())
PARENT_BATCH   = 100
parent_failed  = []

for i in tqdm(range(0, len(parent_list), PARENT_BATCH),
              desc="Upserting parents"):
    batch = parent_list[i:i + PARENT_BATCH]

    # Parents use zero vector — never searched directly
    # Only retrieved by parent_id lookup
    vectors = []
    for parent in batch:
        vectors.append({
            "id"    : parent["parent_id"],
            "values": [0.0] * DIMENSION,
            "metadata": {
                "title"      : parent["title"],
                "source"     : parent["source"],
                "type"       : parent["type"],
                "text"       : parent["text"][:3000],
                "word_count" : parent["word_count"],
                "chunk_type" : "parent",
                "chunk_num"  : parent["chunk_num"]
            }
        })

    try:
        index.upsert(vectors=vectors, namespace="parents")
        time.sleep(0.2)
    except Exception as e:
        print(f"\n  ✗ Parent batch {i} failed: {e}")
        parent_failed.append(i)

# ── Final stats ───────────────────────────────────────────────
stats = index.describe_index_stats()
print(f"\n{'='*55}")
print(f"✅ Parents uploaded!")
print(f"   Parents namespace : "
      f"{stats['namespaces'].get('parents', {}).get('vector_count', 0)}")
print(f"   Children namespace: "
      f"{stats['namespaces'].get('', {}).get('vector_count', 0)}")
print(f"   Total vectors     : {stats['total_vector_count']}")

Upserting 2698 parent chunks...
Namespace: 'parents'



Upserting parents:   4%|▎         | 1/27 [00:00<00:12,  2.02it/s]


  ✗ Parent batch 0 failed: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Mon, 02 Mar 2026 06:58:29 GMT', 'Content-Type': 'application/json', 'Content-Length': '146', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '187', 'x-envoy-upstream-service-time': '60', 'x-pinecone-response-duration-ms': '192', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Dense vectors must contain at least one non-zero value. Vector ID 'parent_cli_CPG_20ABX_0' contains only zeros","details":[]}



Upserting parents:   7%|▋         | 2/27 [00:00<00:10,  2.31it/s]


  ✗ Parent batch 100 failed: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Mon, 02 Mar 2026 06:58:29 GMT', 'Content-Type': 'application/json', 'Content-Length': '157', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '66', 'x-envoy-upstream-service-time': '9', 'x-pinecone-response-duration-ms': '79', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Dense vectors must contain at least one non-zero value. Vector ID 'parent_cli_CPG_20Maagement_20of_8' contains only zeros","details":[]}



...


  ✗ Parent batch 2400 failed: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Mon, 02 Mar 2026 06:58:39 GMT', 'Content-Type': 'application/json', 'Content-Length': '161', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '46', 'x-envoy-upstream-service-time': '4', 'x-pinecone-response-duration-ms': '65', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Dense vectors must contain at least one non-zero value. Vector ID 'parent_drug_Rivastigmine_13_3mg__1377' contains only zeros","details":[]}



Upserting parents:  96%|█████████▋| 26/27 [00:10<00:00,  2.75it/s]


  ✗ Parent batch 2500 failed: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Mon, 02 Mar 2026 06:58:39 GMT', 'Content-Type': 'application/json', 'Content-Length': '161', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '47', 'x-envoy-upstream-service-time': '3', 'x-pinecone-response-duration-ms': '64', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Dense vectors must contain at least one non-zero value. Vector ID 'parent_drug_Sodium_Tetradecyl_Su_1477' contains only zeros","details":[]}



Upserting parents: 100%|██████████| 27/27 [00:11<00:00,  2.44it/s]


  ✗ Parent batch 2600 failed: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Mon, 02 Mar 2026 06:58:39 GMT', 'Content-Type': 'application/json', 'Content-Length': '161', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '5', 'x-envoy-upstream-service-time': '6', 'x-pinecone-response-duration-ms': '26', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Dense vectors must contain at least one non-zero value. Vector ID 'parent_drug_Topiramate_25_mg_Cap_1577' contains only zeros","details":[]}


✅ Parents uploaded!
   Parents namespace : 0
   Children namespace: 0
   Total vectors     : 5708


## **Cell 20 - Final Verification + Download Files**

In [ ]:
print("="*55)
print("FINAL VERIFICATION")
print("="*55)

# Pinecone stats
stats          = index.describe_index_stats()
children_count = stats["namespaces"].get(
    "", {}
).get("vector_count", 0)
parents_count  = stats["namespaces"].get(
    "parents", {}
).get("vector_count", 0)

print(f"\n📊 Pinecone Index: '{INDEX_NAME}'")
print(f"   Children (searchable) : {children_count}")
print(f"   Parents  (context)    : {parents_count}")
print(f"   Total vectors         : {stats['total_vector_count']}")

# Test search
print(f"\n🔍 Test search...")
test_query     = "first line treatment diabetes mellitus"
test_embedding = embedder.encode(
    [test_query],
    normalize_embeddings=True
).tolist()[0]

results = index.query(
    vector          = test_embedding,
    top_k           = 3,
    include_metadata= True,
    namespace       = ""
)

print(f"   Query: '{test_query}'")
for i, match in enumerate(results["matches"]):
    print(f"\n   Result {i+1}:")
    print(f"   Score  : {match['score']:.4f}")
    print(f"   Title  : {match['metadata']['title'][:50]}")
    print(f"   Text   : {match['metadata']['text'][:100]}...")

# Drive files
print(f"\n📁 Files in Drive:")
files_to_check = [
    f"{CLEAN_DIR}/medchatbot_dataset.json",
    f"{CLEAN_DIR}/parents.json",
    f"{CLEAN_DIR}/children.json",
    f"{CLEAN_DIR}/bm25_index.pkl",
    f"{LOG_DIR}/embed_progress.json"
]

for filepath in files_to_check:
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024*1024)
        print(f"   ✅ {Path(filepath).name} ({size_mb:.1f} MB)")
    else:
        print(f"   ❌ {Path(filepath).name} — NOT FOUND")

FINAL VERIFICATION

📊 Pinecone Index: 'medical-knowledge'
   Children (searchable) : 0
   Parents  (context)    : 0
   Total vectors         : 5708

🔍 Test search...
   Query: 'first line treatment diabetes mellitus'

   Result 1:
   Score  : 0.9177
   Title  : Fenofibrate 145mg tablet
   Text   : Drug: Fenofibrate 145mg tablet. MDC Code: C10AB05-000-T10-02-XXX. Category: A/KK. Indications: 1) As...

   Result 2:
   Score  : 0.9166
   Title  : Insulin Degludec/ Insulin Aspart 70/30 Solution fo
   Text   : Drug: Insulin Degludec/ Insulin Aspart 70/30 Solution for Injection in Pre- Filled Pen 100 Units/mL....

   Result 3:
   Score  : 0.9153
   Title  : Dapagliflozin 10mg Tablet
   Text   : Drug: Dapagliflozin 10mg Tablet. MDC Code: A10BX09-999-T32-01-XXX. Category: A/KK. Indications: Indi...

📁 Files in Drive:
   ✅ medchatbot_dataset.json (12.4 MB)
   ✅ parents.json (11.2 MB)
   ✅ children.json (16.8 MB)
   ✅ bm25_index.pkl (36.8 MB)
   ✅ embed_progress.json (0.3 MB)


## **Cell 21 - Downlaod Files Needed for VsCode**

In [ ]:
from google.colab import files

print("Downloading files needed for VSCode backend...\n")

download_files = [
    f"{CLEAN_DIR}/bm25_index.pkl",   # BM25 sparse index
    f"{CLEAN_DIR}/parents.json",     # parent lookup fallback
]

for filepath in download_files:
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024*1024)
        print(f"Downloading: {Path(filepath).name} ({size_mb:.1f} MB)")
        files.download(filepath)
    else:
        print(f"❌ {filepath} not found — check previous cells")

print(f"\n✅ Colab pipeline complete!")
print(f"""
Next steps:
───────────────────────────────────────────
1. Save bm25_index.pkl → put in backend/ folder in VSCode
2. Save parents.json   → put in backend/ folder in VSCode
3. Request MedGemma access on Kaggle/HuggingFace
4. Start building FastAPI backend in VSCode
""")


Downloading: bm25_index.pkl (36.8 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: parents.json (11.2 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Colab pipeline complete!

Next steps:
───────────────────────────────────────────
1. Save bm25_index.pkl → put in backend/ folder in VSCode
2. Save parents.json   → put in backend/ folder in VSCode
3. Request MedGemma access on Kaggle/HuggingFace
4. Start building FastAPI backend in VSCode



## **Cell 25 - Download Files Needed for VsCode**

In [ ]:
from google.colab import files

print("Downloading files needed for VSCode backend...\n")

# These 2 files are needed in VSCode
download_files = [
    f"{CLEAN_DIR}/bm25_index.pkl",   # BM25 sparse index
    f"{CLEAN_DIR}/parents.json",     # parent lookup fallback
]

for filepath in download_files:
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024*1024)
        print(f"Downloading: {Path(filepath).name} ({size_mb:.1f} MB)")
        files.download(filepath)
    else:
        print(f"❌ {filepath} not found — check previous cells")

print(f"\n✅ Colab pipeline complete!")
print(f"""
Next steps:
───────────────────────────────────────────
1. Save bm25_index.pkl → put in backend/ folder in VSCode
2. Save parents.json   → put in backend/ folder in VSCode
3. Request MedGemma access on Kaggle/HuggingFace
4. Start building FastAPI backend in VSCode
""")
```

---

## Summary — All 25 Cells
```
✅ Cell 1  → Mount Drive
✅ Cell 2  → Install scraping libs
✅ Cell 3  → Configuration
✅ Cell 4  → Import libraries
✅ Cell 5  → Helper functions
✅ Cell 6  → Scraper functions
✅ Cell 7  → Scrape + download
✅ Cell 9  → Preprocessing functions
✅ Cell 10 → Process CPG PDFs
✅ Cell 11 → Process FUKKM
✅ Cell 12 → Save dataset
✅ Cell 13 → Quality check
✅ Cell 14 → Visualisation
⬜ Cell 15 → Install chunking libs   ← run now
⬜ Cell 16 → Chunking config
⬜ Cell 17 → Load dataset
⬜ Cell 18 → Parent-child chunking
⬜ Cell 19 → Build BM25 index
⬜ Cell 20 → Load S-PubMedBert
⬜ Cell 21 → Setup Pinecone
⬜ Cell 22 → Embed + upload children
⬜ Cell 23 → Upload parents
⬜ Cell 24 → Verify everything
⬜ Cell 25 → Download for VSCode

SyntaxError: invalid character '✅' (U+2705) (673626923.py, line 34)